# 02 - Preprocesamiento y preparación para modelado

Este cuaderno limpia los datos, construye características útiles y prepara los conjuntos de entrenamiento y prueba para los modelos predictivos.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

DATA_PATH = Path("../data/who_ambient_air_quality_database_version_2024_(v6.1).xlsx")

# Cargar datos
raw_df = pd.read_excel(DATA_PATH, sheet_name="Update 2024 (V6.1)")

# Limpiar y transformar variables numéricas
for col in ["year", "population", "latitude", "longitude", "pm10_concentration", "pm25_concentration", "no2_concentration", "pm10_tempcov", "pm25_tempcov", "no2_tempcov", "who_ms"]:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

# Eliminar filas sin target
clean_df = raw_df.dropna(subset=["pm25_concentration"]).copy()

# Ajustes simples de calidad
clean_df["type_of_stations"] = clean_df["type_of_stations"].fillna("Unknown")
clean_df["country_name"] = clean_df["country_name"].fillna("Unknown")
clean_df["who_region"] = clean_df["who_region"].fillna("Unknown")

# Definir variables de entrada y objetivo
features = [
    "year", "population", "latitude", "longitude",
    "pm10_concentration", "no2_concentration",
    "pm10_tempcov", "no2_tempcov",
    "who_region", "country_name", "type_of_stations"
]
target = "pm25_concentration"

X = clean_df[features]
y = clean_df[target]

# Separación train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Entrenamiento: {X_train.shape}")
print(f"Prueba: {X_test.shape}")

X_train.head()

Entrenamiento: (17384, 11)
Prueba: (4346, 11)


,year,population,latitude,longitude,pm10_concentration,no2_concentration,pm10_tempcov,no2_tempcov,who_region,country_name,type_of_stations
3812,2014.0,297459.0,53.131533,23.189167,27.444,10.988,98.0,90.0,4_Eur,Poland,"Urban, Suburban, Urban"
19753,2018.0,151744.0,26.874850,100.224283,NaN,NaN,NaN,NaN,6_Wpr,China,Unknown
4413,2020.0,968993.0,44.837380,-0.591980,16.781,17.842,95.0,97.0,4_Eur,France,"Urban, Urban, Urban, Urban, Urban"
9764,2020.0,NaN,46.814700,2.610100,NaN,2.074,NaN,100.0,4_Eur,France,Rural
14176,2021.0,542165.0,40.255247,-76.905042,15.000,NaN,NaN,NaN,2_Amr,United States of America,Unknown


In [2]:
# Definir preprocesador
numeric_features = ["year", "population", "latitude", "longitude", "pm10_concentration", "no2_concentration", "pm10_tempcov", "no2_tempcov"]
categorical_features = ["who_region", "country_name", "type_of_stations"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Forma de los datos procesados:")
print(X_train_processed.shape)
print(X_test_processed.shape)

Forma de los datos procesados:
(17384, 412)
(4346, 412)


In [3]:
# Mostrar nombres de columnas del preprocesador
feature_names = []
feature_names.extend(numeric_features)
feature_names.extend(preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features))

print(feature_names[:20])
print(f"Total de características: {len(feature_names)}")

['year', 'population', 'latitude', 'longitude', 'pm10_concentration', 'no2_concentration', 'pm10_tempcov', 'no2_tempcov', 'who_region_1_Afr', 'who_region_2_Amr', 'who_region_3_Sear', 'who_region_4_Eur', 'who_region_5_Emr', 'who_region_6_Wpr', 'who_region_7_NonMS', 'country_name_Albania', 'country_name_Algeria', 'country_name_Argentina', 'country_name_Australia', 'country_name_Austria']
Total de características: 412


In [4]:
# Guardar una versión procesada para reutilizar en el siguiente notebook
processed_df = pd.DataFrame(X_train_processed.toarray(), columns=feature_names)
processed_df["target"] = y_train.reset_index(drop=True)
processed_df.head()

,year,population,latitude,longitude,pm10_concentration,no2_concentration,pm10_tempcov,no2_tempcov,who_region_1_Afr,who_region_2_Amr,...,"type_of_stations_Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Suburban, Urban, Urban, Urban, Suburban","type_of_stations_Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Suburban, Urban, Urban, Urban, Suburban, Urban","type_of_stations_Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban","type_of_stations_Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban","type_of_stations_Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban","type_of_stations_Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Suburban, Urban, Urban, Suburban, Rural","type_of_stations_Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban","type_of_stations_Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban","type_of_stations_Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban, Urban",target
0,-0.929726,-0.216880,0.866570,0.191905,0.102526,-0.815334,0.377644,0.002949,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.723
1,0.446731,-0.287135,-0.783549,1.284646,-0.210194,-0.117548,0.249477,0.247993,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.667
2,1.134960,0.106894,0.345319,-0.145429,-0.387775,-0.089263,0.185394,0.346010,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.618
3,1.134960,-0.218144,0.469585,-0.100008,-0.210194,-1.759628,0.249477,0.493037,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.999
4,1.479074,-0.098897,0.057351,-1.227928,-0.469668,-0.117548,0.249477,0.247993,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.450
